# 2. Camada Bronze - Ingestão dos Dados

## 2.1 Objetivo
Esta etapa tem como objetivo realizar o carregamento dos dados brutos no ambiente Databricks, constituindo a camada Bronze do pipeline.

O conjunto de dados utilizado é o Loan Approval Prediction Dataset. Esse dataset contém informações cadastrais e financeiras relacionadas a solicitações de crédito. O arquivo original foi armazenado em um Volume do Unity Catalog, preservando os dados conforme disponibilizados na fonte.

Nesta etapa não foram realizadas transformações nos dados. O objetivo é manter uma representação dos dados brutos, garantindo sua rastreabilidade para as etapas posteriores do pipeline.

## 2.2 Carregamento dos Dados Brutos
Após o armazenamento do arquivo original no Volume do Unity Catalog, os dados são carregados no notebook utilizando o Apache Spark.

O arquivo CSV é lido diretamente do Volume do Unity Catalog, sendo a primeira linha utilizada como cabeçalho e realizada a identificação automática dos tipos de dados. Neste momento, nenhuma transformação ou tratamento é aplicado aos registros.

A visualização inicial permite verificar se o arquivo foi carregado corretamente.

In [0]:
# --- Caminho do arquivo bruto armazenado no Volume do Unity Catalog ---
caminho_arquivo = "/Volumes/workspace/default/loan_approval_bronze/loan_prediction.csv"

# --- Leitura do arquivo CSV utilizando Apache Spark ---
df_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(caminho_arquivo)
)

# --- Visualização dos primeiros registros ---
display(df_bronze)

_c0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,null,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
5,LP001011,Male,Yes,2,Graduate,Yes,5417,4196.0,267.0,360.0,1.0,Urban,Y
6,LP001013,Male,Yes,0,Not Graduate,No,2333,1516.0,95.0,360.0,1.0,Urban,Y
7,LP001014,Male,Yes,3+,Graduate,No,3036,2504.0,158.0,360.0,0.0,Semiurban,N
8,LP001018,Male,Yes,2,Graduate,No,4006,1526.0,168.0,360.0,1.0,Urban,Y
9,LP001020,Male,Yes,1,Graduate,No,12841,10968.0,349.0,360.0,1.0,Semiurban,N


## 2.3 Verificação Inicial dos Dados Carregados

Após o carregamento, é realizada uma verificação inicial da estrutura dos dados brutos, com o objetivo de conferir a quantidade de registros, atributos e os tipos de dados identificados automaticamente pelo Apache Spark.


In [0]:
# --- Quantidade de registros ---
quantidade_registros = df_bronze.count()

# --- Quantidade de colunas ---
quantidade_colunas = len(df_bronze.columns)

print(f"Quantidade de registros: {quantidade_registros}")
print(f"Quantidade de colunas: {quantidade_colunas}")

# --- Estrutura e tipos de dados identificados ---
df_bronze.printSchema()

Quantidade de registros: 614
Quantidade de colunas: 14
root
 |-- _c0: integer (nullable = true)
 |-- Loan_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Married: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Education: string (nullable = true)
 |-- Self_Employed: string (nullable = true)
 |-- ApplicantIncome: integer (nullable = true)
 |-- CoapplicantIncome: double (nullable = true)
 |-- LoanAmount: double (nullable = true)
 |-- Loan_Amount_Term: double (nullable = true)
 |-- Credit_History: double (nullable = true)
 |-- Property_Area: string (nullable = true)
 |-- Loan_Status: string (nullable = true)



### 2.3.1 Resultado da Verificação Inicial

A verificação identificou 614 registros e 14 colunas na estrutura carregada pelo Apache Spark.

Dessas 14 colunas, 13 correspondem aos atributos do conjunto de dados utilizado na análise de crédito. A coluna adicional _c0 representa um índice sequencial associado aos registros do arquivo de origem e não possui significado para o contexto do problema.

Também foi possível verificar os tipos de dados identificados automaticamente pelo Spark. Atributos categóricos, como Gender, Married e Education , foram reconhecidos corretamente como texto (string), enquanto atributos numéricos, como ApplicantIncome, CoapplicantIncome e LoanAmount, foram identificados corretamente como tipos numéricos.

A coluna _c0 foi mantida nesta etapa para preservar a estrutura dos dados na camada Bronze e removida posteriormente durante o processo de tratamento e preparação da camada Silver.

## 2.4 Persistência dos Dados da Camada Bronze

Após o carregamento e a verificação inicial, os dados brutos são persistidos no Unity Catalog como uma tabela no formato Delta.

A persistência permite que os dados permaneçam armazenados na plataforma e possam ser acessados posteriormente pelas demais etapas do pipeline, sem depender exclusivamente do DataFrame criado durante a execução do notebook.

In [0]:
# --- Persistência dos dados da camada Bronze ---

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.loan_approval_bronze_raw")
)

print("Tabela Bronze persistida com sucesso.")

Tabela Bronze persistida com sucesso.


In [0]:
# --- Validação da tabela Bronze persistida ---

df_bronze_validacao = spark.table("workspace.default.loan_approval_bronze_raw")

print(f"Registros persistidos: {df_bronze_validacao.count()}")
print(f"Atributos persistidos: {len(df_bronze_validacao.columns)}")

display(df_bronze_validacao.limit(10))

Registros persistidos: 614
Atributos persistidos: 14


_c0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,null,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
5,LP001011,Male,Yes,2,Graduate,Yes,5417,4196.0,267.0,360.0,1.0,Urban,Y
6,LP001013,Male,Yes,0,Not Graduate,No,2333,1516.0,95.0,360.0,1.0,Urban,Y
7,LP001014,Male,Yes,3+,Graduate,No,3036,2504.0,158.0,360.0,0.0,Semiurban,N
8,LP001018,Male,Yes,2,Graduate,No,4006,1526.0,168.0,360.0,1.0,Urban,Y
9,LP001020,Male,Yes,1,Graduate,No,12841,10968.0,349.0,360.0,1.0,Semiurban,N


### 2.4.1 Resultado da Persistência

A tabela loan_approval_bronze_raw foi persistida com sucesso no Unity Catalog utilizando o formato Delta.

Com a persistência, os dados da camada Bronze passam a permanecer armazenados no ambiente Databricks e podem ser utilizados pelas etapas posteriores do pipeline sem depender exclusivamente do DataFrame criado durante a execução do notebook.